In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions.lf_clustering import s10_load_ersps
from functions.lf_minus101 import (
    s23_build_minus101_feature_matrix,
    q60_plot_minus101_overlays,
    q61_plot_minus101_prototype,
)
from functions.lf_blob_metrics import s22_build_blob_feature_matrix
from functions.lf_clustering_methods import (
    s53_compute_medoids,
    s54_compute_prototypes,
    q52_plot_prototype_grid,
)
from functions import lf_cluster_run as R

SCRIPT_NAME = '231_minus101_clustering.ipynb'


## Config

## Load canonical dataset (shared across 210/230/231/232)

In [2]:
# ── Canonical dataset (shared by 210/230/231/232) ──────────────────
# Loads ERSPs from INPUT_DIR, drops non-neural channels, then gates by
# high-activity. SAME filter for every clustering notebook so cross-
# feature-set / cross-method comparisons are on the IDENTICAL sample set.
# Cached in 02_FBM_Clustering/outputs/_dataset/canonical/ — subsequent
# notebook runs load instantly instead of re-walking the ERSP_matrix tree.
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

df_meta, ersp_list, X_3d = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d.shape={X_3d.shape}')


[lf_dataset cache hit] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\canonical
  1538 samples · X_3d.shape=(1538, 129, 300)

Canonical dataset: 1538 samples · X_3d.shape=(1538, 129, 300)


## Build blobs on the canonical samples (no 230 dependency)

In [3]:
from functions.lf_blob_metrics import s22_build_blob_feature_matrix
from functions import lf_blob_clustering_config as bcfg

VALLEY_PARAMS = bcfg.VALLEY_PARAMS

_, _, blobs_per_sample = s22_build_blob_feature_matrix(
    ersp_list=ersp_list,
    max_blobs=int(VALLEY_PARAMS['max_blobs']),
    thr_pos=float(VALLEY_PARAMS['thr_pos']),
    thr_neg=float(VALLEY_PARAMS['thr_neg']),
    delta_valley=float(VALLEY_PARAMS['delta_valley']),
    min_mean_pos=float(VALLEY_PARAMS['min_mean_pos']),
    max_mean_neg=float(VALLEY_PARAMS['max_mean_neg']),
    sign_mode=str(VALLEY_PARAMS['sign_mode']),
)
print('blobs_per_sample built:', len(blobs_per_sample))


Valley-blob feature matrix: X_blob.shape=(1538, 48) (max_blobs=6, features_per_blob=8)
blobs_per_sample built: 1538


In [5]:
# sorted(RUNS_DIR_230.glob('*'))
# BLOB_RUNS_DIR

## 3 — Build -101 feature matrix

In [6]:
X_101, ds_shape = s23_build_minus101_feature_matrix(
    ersp_list=ersp_list,
    blobs_per_sample=blobs_per_sample,
    scale=SCALE,
    score_min=SCORE_MIN,
)
print('X_101:', X_101.shape, '  ds_shape:', ds_shape)

NameError: name 'SCALE' is not defined

## 4 — QC: painted -101 maps

In [ ]:
# Sample 30 random indices for visual inspection
rng = np.random.default_rng(42)
qc_idx = rng.choice(len(ersp_list), size=min(30, len(ersp_list)), replace=False).tolist()

q60_plot_minus101_overlays(
    indices=qc_idx,
    ersp_list=ersp_list,
    blobs_per_sample=blobs_per_sample,
    df_meta=df_keep,
    scale=SCALE,
    score_min=SCORE_MIN,
)

# Clustering

Two methods on the same `X_101` (downsampled -1 / 0 / +1 painted maps, flattened): K-Means (K-sweep) and Hierarchical.
Each `fit_and_save` call writes a self-contained run directory and updates `outputs/clustering/index.json`.


In [ ]:
# KMeans K-sweep on minus101 features.
manifest_km = R.fit_and_save(
    X_101,
    df_keep=df_meta,
    method='kmeans',
    feature_set='minus101',
    params={'k_range': KMEANS_K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means',
    feature_set_label='\u22121 / 0 / +1 Segmentation',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f'Best K (KMeans/minus101, by silhouette): {BEST_K}')


In [ ]:
# Hierarchical on minus101 — K-sweep so MOBA can scrub.
manifest_hc = R.fit_and_save(
    X_101,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='minus101',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': KMEANS_K_RANGE},
    method_label='Hierarchical',
    feature_set_label='\u22121 / 0 / +1 Segmentation',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/minus101, by silhouette): {manifest_hc["summary"]["best_k"]}')


## Per-cluster centroid PNGs (for the MOBA cluster chips)

Walks `index.json`, for every `feature_set == 'minus101'` run, computes the mean -1/0/+1 map
per cluster (using `X_101` reshaped back to `ds_shape`) and writes
`<run_dir>/cluster_centroids/cluster_<NN>.png`. The MOBA chip thumbnails update automatically.


In [ ]:
# BACKFILL_CENTROIDS — per-cluster mean -1/0/+1 thumbnails for the MOBA chips (minus101 feature_set).
import json

INDEX_PATH = CLUSTERING_DIR / 'index.json'

def _save_per_cluster_centroid_pngs_minus101(manifest, X_local, ds_shape_local, *, vlim=1.0):
    if manifest['feature_set'] != 'minus101':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    df = pd.read_csv(run_dir / 'labels.csv')
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    if cluster_col not in df.columns:
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]
    labels = df[cluster_col].to_numpy()
    if len(labels) != X_local.shape[0]:
        print(f"  [skip] {manifest['run_id']}: labels ({len(labels)}) vs X_101 ({X_local.shape[0]}) mismatch — re-run cells 5–10 in this notebook with the same gating as the run")
        return 0
    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)
    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        mean_map = X_local[idx].mean(axis=0).reshape(ds_shape_local)  # (ds_freq, ds_time) ≈ (13, 30)
        fig, ax = plt.subplots(figsize=(2, 1.5))
        ax.imshow(mean_map, aspect='auto', origin='lower',
                  cmap='bwr', vmin=-vlim, vmax=vlim, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        out_path = out_dir / f'cluster_{int(c):02d}.png'
        fig.savefig(out_path, dpi=80, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)

if not INDEX_PATH.exists():
    print(f'No {INDEX_PATH} yet — run the fit_and_save cells above first.')
else:
    with open(INDEX_PATH) as f:
        idx_data = json.load(f)
    runs = [r for r in idx_data.get('runs', []) if r['feature_set'] == 'minus101']
    print(f'Backfilling per-cluster centroid PNGs for {len(runs)} minus101 runs...')
    for run in runs:
        manifest_path = CLUSTERING_DIR / run['path'] / 'manifest.json'
        if not manifest_path.exists():
            print(f"  [skip] {run['path']}: missing manifest.json"); continue
        with open(manifest_path) as f:
            manifest = json.load(f)
        n = _save_per_cluster_centroid_pngs_minus101(manifest, X_101, ds_shape)
        if n:
            print(f"  [{manifest['method']}/{manifest['feature_set']}] {manifest['run_id']}  ->  {n} cluster PNGs")
    print('\nDone. Commit and push:')
    print('  git add 02_FBM_Clustering/outputs/clustering')
    print('  git commit -m "Per-cluster centroid PNGs for MOBA chips (minus101)"')
    print('  git push')
